# Session 3 Explorer: Adversarial Testing

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/buildLittleWorlds/level-2-course-material/blob/main/session-03/explorer.ipynb)

*Apply this week's method to a model from your Collection*

## What This Notebook Is For

In class you broke a sentiment model with sarcasm, emoji, and ALL CAPS -- then built a `clean_text()` function to fix what you could. You also learned that some failures are **noise problems** (fixable) and some are **meaning problems** (unfixable).

Now pick a model from **your Collection** and try to break it the same way. Design adversarial inputs, run them raw, run them cleaned, and figure out what kind of failures you're seeing.

By the end you'll have:
- 5 adversarial inputs tested raw and cleaned
- A before/after comparison
- A draft Research Journal entry
- A tasting note about how your model handles messy input

In [ ]:
!pip install -q transformers torch
from transformers import pipeline
import re
print("Setup complete!")

## Load YOUR Model

Pick a model from your Collection. It can be the same one you tested in the Session 2 Explorer or a different one.

Replace `YOUR-MODEL-NAME-HERE` with the model name from Hugging Face.

In [ ]:
# CHANGE THIS: Replace with a model name from your Collection
# Examples:
#   "distilbert-base-uncased-finetuned-sst-2-english"
#   "cardiffnlp/twitter-roberta-base-sentiment-latest"
#   "j-hartmann/emotion-english-distilroberta-base"

my_model = pipeline("text-classification", model="YOUR-MODEL-NAME-HERE")
print("Your model loaded!")

## The clean_text() Function

This is the same function from class. Run this cell to define it -- nothing to change.

In [ ]:
def clean_text(text):
    # 1. Strip leading/trailing whitespace
    text = text.strip()

    # 2. Collapse multiple spaces into one
    text = re.sub(r' {2,}', ' ', text)

    # 3. Limit repeated characters ("sooooo" -> "soo")
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)

    # 4. Remove emoji
    text = re.sub(
        r'[\U0001F600-\U0001F64F'
        r'\U0001F300-\U0001F5FF'
        r'\U0001F680-\U0001F6FF'
        r'\U00002600-\U000026FF]+',
        ' ', text
    )

    # 5. Normalize ALL CAPS to Title Case
    words = text.split()
    caps_count = sum(1 for w in words if w.isupper() and len(w) > 1)
    if caps_count > 3:
        text = text.title()

    # Clean up any extra spaces from cleaning
    text = re.sub(r' {2,}', ' ', text).strip()

    return text

print("clean_text() is ready!")

## Design Your 5 Adversarial Inputs

Try to **break** your model. Think about what confused the class model:
- Sarcasm and irony
- Emoji that contradict the words
- ALL CAPS for emphasis or tone
- Repeated characters ("soooooo")
- Sentences that could be sincere or sarcastic

Replace the example texts below with your own adversarial inputs.

In [ ]:
# CHANGE THESE: Design 5 inputs meant to confuse or break the model
adversarial_inputs = [
    "Replace this with a sarcastic sentence.",
    "Replace this with emoji that contradict the words.",
    "Replace this with ALL CAPS emphasis.",
    "Replace this with repeated characters or internet slang.",
    "Replace this with something that could be sincere OR sarcastic.",
]

print(f"Ready to test {len(adversarial_inputs)} adversarial inputs.")

## Run Model on Raw Inputs

First, see what the model does with your messy inputs -- no cleaning.

In [ ]:
print("RAW INPUTS (no cleaning):")
print("=" * 55)

raw_results = []
for i, text in enumerate(adversarial_inputs, 1):
    result = my_model(text)[0]
    raw_results.append(result)
    print(f"  {i}. {result['label']} ({result['score']:.0%})")
    print(f"     \"{text}\"")
    print()

## Run Model on Cleaned Inputs

Now run the same inputs through `clean_text()` first, then through the model.

In [ ]:
print("CLEANED INPUTS:")
print("=" * 55)

cleaned_results = []
for i, text in enumerate(adversarial_inputs, 1):
    cleaned = clean_text(text)
    result = my_model(cleaned)[0]
    cleaned_results.append(result)
    print(f"  {i}. {result['label']} ({result['score']:.0%})")
    print(f"     Original: \"{text}\"")
    print(f"     Cleaned:  \"{cleaned}\"")
    print()

## Side-by-Side Comparison

This table shows before and after for every input so you can spot the differences.

In [ ]:
print(f"{'#':<4} {'RAW':<20} {'CLEANED':<20} {'CHANGED?':<10}")
print("-" * 54)

noise_fixes = 0
no_change = 0

for i, (raw, cleaned) in enumerate(zip(raw_results, cleaned_results), 1):
    changed = raw['label'] != cleaned['label']
    status = "YES" if changed else "no"
    if changed:
        noise_fixes += 1
    else:
        no_change += 1
    
    raw_str = f"{raw['label']} ({raw['score']:.0%})"
    cleaned_str = f"{cleaned['label']} ({cleaned['score']:.0%})"
    print(f"  {i:<2} {raw_str:<20} {cleaned_str:<20} {status:<10}")

print()
print(f"Cleaning changed the label on {noise_fixes} out of {len(adversarial_inputs)} inputs.")
print(f"{no_change} inputs were unaffected by cleaning.")

## Reflect: Noise vs. Meaning

Look at your results and sort each failure:

**Which failures were noise problems?** (Cleaning fixed them -- the problem was formatting, not meaning.)

*Write here:*

**Which failures were meaning problems?** (Cleaning didn't help -- the model can't read sarcasm, irony, or tone.)

*Write here:*

**Did your model handle adversarial input better or worse than the class model?** Why do you think that is?

*Write here:*

## Research Journal Draft

Fill in the sections below. This becomes part of your Research Journal.

---

### Week 3: Adversarial Testing

**This Week's Method:**
Adversarial testing -- deliberately designing inputs to break a model, then using data cleaning to separate noise failures from meaning failures.

**How I Applied It:**
*Which model did you test? What kinds of adversarial inputs did you design?*



**What I Expected:**
*Before you ran the code, which inputs did you think would break the model?*



**What I Found:**
*What actually broke the model? What survived? Did cleaning help?*



**Why I Think This Happened:**
*What does this tell you about what the model learned from its training data?*



**Limitations:**
*What couldn't you figure out from this test? What would you need to test next?*



**What I Want to Try Next:**
*What new question did this experiment give you?*



## Collection Tasting Note

Write a tasting note focused on how this model handles messy and adversarial input. Add this to your Collection on Hugging Face.

---

**Model name:**

**How it handles sarcasm:**

**How it handles emoji:**

**How it handles ALL CAPS:**

**Biggest weakness I found:**

**Does cleaning help?** (yes / no / sometimes)

**I'd trust this model with:**

**I would NOT trust this model with:**

---

*Copy this tasting note into your Collection description on Hugging Face.*